In [1]:
import numpy as np
from sklearn.datasets import fetch_openml 
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
mnist = fetch_openml("mnist_784",version=1,as_frame=True)
X = mnist["data"]
y = mnist["target"]


In [3]:
print(mnist.target_names)

['class']


In [4]:
data = pd.DataFrame(X)
y = pd.DataFrame(y)

Data = pd.concat([y,data],axis=1)
Data.shape

(70000, 785)

In [31]:
data = np.array(Data,dtype = int)
m,n = data.shape
np.random.shuffle(data)

data_dev = data[0:10000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]
X_dev = X_dev/ 255.

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]
X_train = X_train / 255.

X_train = np.astype(X_train,int)
Y_dev = Y_dev.astype(int)




In [38]:
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(784, 69000))

In [ ]:
def init_params():
    W1 = np.random.rand(10,784) - 0.5
    B1 = np.random.rand(10,1) -0.5
    W2 = np.random.rand(10,10)-0.5
    B2 = np.random.rand(10,1)  -0.5
    
    return W1,W2, B1, B2

def relu(layer):
    return np.maximum(0,layer)

def softmax(layer):
    return np.exp(layer) / np.sum(np.exp(layer))
    
    
def forward(W1,W2,B1,B2,X):
    first = W1.dot(X) + B1 
    a1 = relu(first)
    second = W2.dot(a1) + B2
    a2 = softmax(second)
    return first, second, a1,a2
    
def onehot(Y):
    one_hot_Y = np.zeros((Y.size,Y.max()+1))
    one_hot_Y[np.arange(Y.size),Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def deriv_Relu(Z):
    return Z > 0

def backprop(second, a2, first, a1, W2, Y, X):
    m = Y.size
    one_hot_Y = onehot(Y)
    dsecond = a2 - one_hot_Y
    dw2= 1/m * dsecond.dot(a1.T)
    db2 = 1/m * np.sum(dsecond)
    dfirst = W2.T.dot(dsecond) * deriv_Relu(first)
    
    dw1= 1/m * dfirst.dot(X.T)
    db1 = 1/m * np.sum(dfirst)
    
    return db1 , db2, dw1, dw2 
    
def update_params(W1, W2, B1, B2, dw1, dw2, db1, db2, alpha):
    W1 = W1 - alpha* dw1
    W2 = W2 - alpha * dw2
    B1 = B1 - alpha * db1
    B2 = B2 - alpha * db2 
    return W1, W2, B1, B2 
    
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions ==Y) / Y.size 

def gradient_descent(X,Y,iterations,alpha):
    W1 , W2, B1, B2 = init_params()
    for i in range(iterations):
        Z1, Z2, A1, A2 = forward(W1, W2,B1,B2,X)
        db1, db2 , dw1, dw2 = backprop(Z2,A2,Z1,A1,W2,Y, X)
        W1,W2,B1,B2 = update_params(W1, W2,B1,B2, dw1,dw2, db1,db2,alpha)
        if i% 10 == 0:
            print("Iteration", i)
            print("Accuracy", get_accuracy(get_predictions(A2),Y))
    return W1,B1,W2,B2     
    


In [37]:
W1 , B1, W2, B2 = gradient_descent(X_train,Y_train,100,0.1)

Iteration 0
[1 4 9 ... 8 4 8] [3 7 4 ... 5 2 5]
Accuracy 0.10188405797101449


C:\Users\akshi\AppData\Local\Temp\ipykernel_13976\271490713.py:13: RuntimeWarning: overflow encountered in exp
  return np.exp(layer) / np.sum(np.exp(layer))
C:\Users\akshi\AppData\Local\Temp\ipykernel_13976\271490713.py:13: RuntimeWarning: invalid value encountered in divide
  return np.exp(layer) / np.sum(np.exp(layer))


Iteration 10
[0 0 0 ... 0 0 0] [3 7 4 ... 5 2 5]
Accuracy 0.09863768115942029
Iteration 20
[0 0 0 ... 0 0 0] [3 7 4 ... 5 2 5]
Accuracy 0.09863768115942029
Iteration 30
[0 0 0 ... 0 0 0] [3 7 4 ... 5 2 5]
Accuracy 0.09863768115942029


KeyboardInterrupt: 